# AMPS II · Week 02 — Lab
## Plane stress, orthotropic plates, stress concentration, axisymmetry

| | |
|---|---|
| **Lecture sections** | Part A (plane stress, orthotropy, Kirsch/Lekhnitskii, axisymmetry, Lamé) · Part B (Gmsh, symmetry conditions, stress recovery) · Part C (axisymmetric code) · Part D (LLM) |
| **Time** | ≈ 80 min in class |
| **You will** | build a plate with a hole in Gmsh and check the mesh · compare plane stress and plane strain · measure $K_t$ against Kirsch and Lekhnitskii · see what the hole geometry order does · solve the Lamé cylinder in $(r,z)$ and measure convergence rates |

**Reminder.** `%%fenicsx` cells are separate MPI programs; ordinary cells have NumPy and matplotlib only. The two model modules are written once
with `%%writefile` (§1) and imported by every `%%fenicsx` cell. Results pass through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The model modules  *(lecture §B1–B4, §C1)*
`hole.py`: the Gmsh plate with a hole (quarter or full), plane-stress isotropic and orthotropic $\boldsymbol\sigma(\mathbf u)$, a solver that returns
$K_t=\max\sigma_{xx}/T$. `axi.py`: the $(r,z)$ mesh, the axisymmetric strain and stress, a general solver, the Lamé solution and the residual on a
boundary. Read them once; run both cells — they only write files.

In [ ]:
%%writefile /tmp/fnx/hole.py
"""Plate with a circular hole (radius R) under tension T along x. Plane stress, isotropic or orthotropic.
quarter model: [0, L]^2 minus the disk, symmetry conditions on x = 0 and y = 0;  full model: [-L, L]^2 minus the disk."""
import json, os
import numpy as np
import ufl
from mpi4py import MPI
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD
os.makedirs("/tmp/fnx", exist_ok=True)
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

LEFT, BOTTOM, RIGHT, TOP, HOLE = 1, 2, 3, 4, 5

def plate_mesh(R=0.1, L=1.0, h_hole=0.005, h_far=None, order=2, full=False):
    """Gmsh on rank 0 -> DOLFINx (API sheet §2). order=2: curved (isoparametric) triangles on the hole.
    Facet tags: 1 left, 2 bottom, 3 right, 4 top, 5 hole. Returns (msh, facet_tags)."""
    import gmsh
    from dolfinx.io import gmsh as gmshio
    h_far = h_far or 0.1 * L
    gmsh.initialize(); gmsh.option.setNumber("General.Verbosity", 0)
    if comm.rank == 0:
        occ = gmsh.model.occ
        x0 = -L if full else 0.0
        s = occ.addRectangle(x0, x0, 0, L - x0, L - x0); d = occ.addDisk(0, 0, 0, R, R)
        occ.cut([(2, s)], [(2, d)])                          # removeObject/removeTool default True: nothing left over
        occ.synchronize()
        surf = gmsh.model.getEntities(2); gmsh.model.addPhysicalGroup(2, [t for _, t in surf], 1)
        tags = {k: [] for k in (LEFT, BOTTOM, RIGHT, TOP, HOLE)}
        for _, c in gmsh.model.getBoundary(surf, oriented=False):
            cx, cy, _ = occ.getCenterOfMass(1, c)
            k = (LEFT if np.isclose(cx, x0) else BOTTOM if np.isclose(cy, x0) else
                 RIGHT if np.isclose(cx, L) else TOP if np.isclose(cy, L) else HOLE)
            tags[k].append(c)
        for k, v in tags.items():
            gmsh.model.addPhysicalGroup(1, v, k)
        f = gmsh.model.mesh.field                            # element size: h_hole on the hole, growing to h_far
        dist = f.add("Distance"); f.setNumbers(dist, "CurvesList", tags[HOLE])
        th = f.add("Threshold"); f.setNumber(th, "InField", dist)
        f.setNumber(th, "SizeMin", h_hole); f.setNumber(th, "SizeMax", h_far)
        f.setNumber(th, "DistMin", 0.0); f.setNumber(th, "DistMax", 0.5 * L)
        f.setAsBackgroundMesh(th)
        for o in ("Mesh.MeshSizeExtendFromBoundary", "Mesh.MeshSizeFromPoints", "Mesh.MeshSizeFromCurvature"):
            gmsh.option.setNumber(o, 0)
        gmsh.model.mesh.generate(2)
        if order == 2:
            gmsh.model.mesh.setOrder(2)
    md = gmshio.model_to_mesh(gmsh.model, comm, 0, gdim=2)
    gmsh.finalize()
    return md.mesh, md.facet_tags

def area(msh):
    return comm.allreduce(fem.assemble_scalar(fem.form(1.0 * ufl.dx(domain=msh))), op=MPI.SUM)

eps = lambda u: ufl.sym(ufl.grad(u))

def isotropic(E=1.0, nu=0.3, plane="stress"):
    """sigma(u) in 2D: plane stress uses lambda* = 2 lambda mu / (lambda + 2 mu) = E nu / (1 - nu^2); plane strain uses lambda"""
    mu = E / (2 * (1 + nu)); lmbda = E * nu / ((1 + nu) * (1 - 2 * nu))
    lam = 2 * lmbda * mu / (lmbda + 2 * mu) if plane == "stress" else lmbda
    return lambda u: 2 * mu * eps(u) + lam * ufl.tr(eps(u)) * ufl.Identity(2)

def orthotropic(E1, E2, nu12, G12, theta=0.0):
    """plane-stress orthotropic sigma(u); material axis 1 at angle theta (radians) from x.
    C = S^-1 in Voigt notation (engineering shear strain 2 eps_12); rotate the strain in, the stress out."""
    S = np.array([[1 / E1, -nu12 / E1, 0], [-nu12 / E1, 1 / E2, 0], [0, 0, 1 / G12]])
    C = ufl.as_matrix(np.linalg.inv(S).tolist())
    c, s = np.cos(theta), np.sin(theta)
    Q = ufl.as_matrix([[c, -s], [s, c]])                     # columns = material axes 1, 2 in (x, y)
    def sigma(u):
        e = Q.T * eps(u) * Q                                   # strain in material axes
        sv = C * ufl.as_vector([e[0, 0], e[1, 1], 2 * e[0, 1]])
        return Q * ufl.as_matrix([[sv[0], sv[2]], [sv[2], sv[1]]]) * Q.T
    return sigma

def lekhnitskii(E1, E2, nu12, G12):
    """Kt of an INFINITE orthotropic plate with a circular hole, tension along material axis 1 (isotropic: 3)"""
    return 1 + np.sqrt(2 * (np.sqrt(E1 / E2) - nu12) + E1 / G12)

def solve_plate(msh, ft, sigma, T=1.0, deg=2, full=False, L=1.0):
    """tension T along x.
    quarter: u_x = 0 on x = 0, u_y = 0 on y = 0 (symmetry), traction (T, 0) on x = L.
    full:    traction (+-T, 0) on x = +-L; three pinned DOFs at (-L, 0) and (L, 0) remove the rigid-body motions
             (the load is self-equilibrated, so the pins carry no force).
    Returns uh and info: Kt = max sigma_xx / T (sigma(u_h) interpolated into DG(deg-1)), the stress field s, DOFs, cells."""
    V = fem.functionspace(msh, ("Lagrange", deg, (2,)))
    ds = ufl.Measure("ds", domain=msh, subdomain_data=ft)
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    t = fem.Constant(msh, np.array([T, 0.0]))
    a = ufl.inner(sigma(u), eps(v)) * ufl.dx
    if full:
        Lf = ufl.dot(t, v) * ds(RIGHT) - ufl.dot(t, v) * ds(LEFT)
        pL = mesh.locate_entities_boundary(msh, 0, lambda x: np.isclose(x[0], -L) & np.isclose(x[1], 0.0))
        pR = mesh.locate_entities_boundary(msh, 0, lambda x: np.isclose(x[0], L) & np.isclose(x[1], 0.0))
        bcs = [fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(i), 0, pL), V.sub(i)) for i in (0, 1)]
        bcs.append(fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(1), 0, pR), V.sub(1)))
    else:
        Lf = ufl.dot(t, v) * ds(RIGHT)
        bcs = [fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(0), 1, ft.find(LEFT)), V.sub(0)),     # u_x = 0
               fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(1), 1, ft.find(BOTTOM)), V.sub(1))]   # u_y = 0
    pr = LinearProblem(a, Lf, bcs=bcs, petsc_options_prefix="plate_",
                       petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"})
    uh = pr.solve()
    W = fem.functionspace(msh, ("DG", max(deg - 1, 0), (2, 2)))
    s = fem.Function(W); s.interpolate(fem.Expression(sigma(uh), W.element.interpolation_points))
    sxx = s.x.array.reshape(-1, 4)[:, 0]
    Kt = comm.allreduce(sxx.max() if sxx.size else -np.inf, op=MPI.MAX) / T
    umax = comm.allreduce(np.abs(uh.x.array).max(), op=MPI.MAX)
    return uh, dict(Kt=Kt, s=s, umax=umax, reason=pr.solver.getConvergedReason(),
                    dofs=V.dofmap.index_map.size_global * 2, cells=msh.topology.index_map(2).size_global)

def argmax_angle(s):
    """polar angle (degrees) of the DG node with the largest sigma_xx"""
    X = s.function_space.tabulate_dof_coordinates(); sxx = s.x.array.reshape(-1, 4)[:, 0]
    k = int(np.argmax(sxx)) if sxx.size else 0
    best = comm.allreduce((float(sxx[k]) if sxx.size else -np.inf, float(np.degrees(np.arctan2(X[k, 1], X[k, 0])))), op=MPI.MAX)
    return best[1]

In [ ]:
%%writefile /tmp/fnx/axi.py
"""Axisymmetric linear elasticity in the (r, z) half-plane, u = (u_r, u_z), d/dtheta = 0."""
import json, os
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, assemble_matrix, assemble_vector

comm = MPI.COMM_WORLD
os.makedirs("/tmp/fnx", exist_ok=True)
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

INNER, OUTER, BOTTOM, TOP = 1, 2, 3, 4
E, nu = 1.0, 0.3
mu = E / (2 * (1 + nu)); lmbda = E * nu / ((1 + nu) * (1 - 2 * nu))

def rz_mesh(ri, ro, h, nr, nz):
    """triangles on [ri, ro] x [0, h]. Facet tags: 1 r = ri (the axis if ri = 0), 2 r = ro, 3 z = 0, 4 z = h"""
    msh = mesh.create_rectangle(comm, [[ri, 0.0], [ro, h]], [nr, nz], cell_type=mesh.CellType.triangle)
    marks = [(INNER, lambda x: np.isclose(x[0], ri)), (OUTER, lambda x: np.isclose(x[0], ro)),
             (BOTTOM, lambda x: np.isclose(x[1], 0.0)), (TOP, lambda x: np.isclose(x[1], h))]
    fac, val = [], []
    for tag, f in marks:
        e = mesh.locate_entities_boundary(msh, 1, f); fac.append(e); val.append(np.full(len(e), tag, dtype=np.int32))
    fac, idx = np.unique(np.concatenate(fac), return_index=True)
    return msh, mesh.meshtags(msh, 1, fac, np.concatenate(val)[idx])

def eps_axi(u, r):
    """3D small strain in cylindrical components (rr, tt, zz): the hoop strain u_r / r is the new entry"""
    g = (u[0].dx(1) + u[1].dx(0)) / 2
    return ufl.as_matrix([[u[0].dx(0), 0, g], [0, u[0] / r, 0], [g, 0, u[1].dx(1)]])

def sigma_axi(u, r):
    e = eps_axi(u, r)
    return 2 * mu * e + lmbda * ufl.tr(e) * ufl.Identity(3)

def solve_axi(msh, ft, deg=2, tractions=(), body=None, fixed=()):
    """tractions: [(tag, (t_r, t_z))] (UFL or numbers); body: (f_r, f_z) or None; fixed: [(tag, component)] -> u_comp = 0.
    Returns uh and info with the forms a, L (needed for reactions) and r."""
    V = fem.functionspace(msh, ("Lagrange", deg, (2,)))
    r = ufl.SpatialCoordinate(msh)[0]
    ds = ufl.Measure("ds", domain=msh, subdomain_data=ft)
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    a = ufl.inner(sigma_axi(u, r), eps_axi(v, r)) * r * ufl.dx
    Lf = ufl.dot(fem.Constant(msh, np.zeros(2)), v) * r * ufl.dx
    for tag, t in tractions:
        Lf += ufl.dot(ufl.as_vector(t), v) * r * ds(tag)
    if body is not None:
        Lf += ufl.dot(ufl.as_vector(body), v) * r * ufl.dx
    bcs = [fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(c), 1, ft.find(tag)), V.sub(c)) for tag, c in fixed]
    pr = LinearProblem(a, Lf, bcs=bcs, petsc_options_prefix="axi_",
                       petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"})
    uh = pr.solve()
    return uh, dict(a=a, L=Lf, r=r, ds=ds, msh=msh, ft=ft, reason=pr.solver.getConvergedReason(),
                    dofs=V.dofmap.index_map.size_global * 2)

def lame(r, ri, ro, pi, po=0.0, ends="open"):
    """thick cylinder (works with numbers and with UFL r): sigma_rr, sigma_tt, sigma_zz, u_r.
    ends: 'open' (sigma_zz = 0), 'closed' (end caps carry the pressure), 'plane_strain' (eps_zz = 0)"""
    A = (pi * ri**2 - po * ro**2) / (ro**2 - ri**2); B = (pi - po) * ri**2 * ro**2 / (ro**2 - ri**2)
    srr, stt = A - B / r**2, A + B / r**2
    szz = {"open": 0.0, "closed": A, "plane_strain": 2 * nu * A}[ends]
    return srr, stt, szz, r / E * (stt - nu * (srr + szz))

def solve_cylinder(n, deg, ri=1.0, ro=2.0, h=1.0, p=1.0, ends="open"):
    """internal pressure p, u_z = 0 on z = 0; 'plane_strain' also fixes u_z on z = h (open: top free).
    Returns uh, info with the r-weighted relative L2 error of u_r against Lame and the mean u_r on the inner wall."""
    msh, ft = rz_mesh(ri, ro, h, n, max(1, n // 2))
    fixed = [(BOTTOM, 1)] + ([(TOP, 1)] if ends == "plane_strain" else [])
    uh, info = solve_axi(msh, ft, deg, tractions=[(INNER, (p, 0.0))], fixed=fixed)   # internal pressure p
    r = info["r"]; ure = lame(r, ri, ro, p, 0.0, ends)[3]
    num = comm.allreduce(fem.assemble_scalar(fem.form((uh[0] - ure) ** 2 * r * ufl.dx)), op=MPI.SUM)
    den = comm.allreduce(fem.assemble_scalar(fem.form(ure ** 2 * r * ufl.dx)), op=MPI.SUM)
    ur_in = comm.allreduce(fem.assemble_scalar(fem.form(uh[0] * info["ds"](INNER))), op=MPI.SUM) / h
    return uh, dict(info, err=np.sqrt(num / den), ur_in=ur_in)

def residual_on(uh, info, tag):
    """r = K u - f with K, f assembled WITHOUT boundary conditions (API sheet §16c); returns (coordinates, r) of the
    DOFs on the facets 'tag' owned by this rank. These are the r-weighted equations: multiply by 2 pi for a force."""
    V = uh.function_space
    A0 = assemble_matrix(fem.form(info["a"])); A0.assemble()
    f0 = assemble_vector(fem.form(info["L"])); f0.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    res = A0.createVecLeft(); A0.mult(uh.x.petsc_vec, res); res.axpy(-1.0, f0)
    dofs = fem.locate_dofs_topological(V, 1, info["ft"].find(tag)); dofs = dofs[dofs < V.dofmap.index_map.size_local]
    return V.tabulate_dof_coordinates()[dofs], res.array.reshape(-1, 2)[dofs]

## 2 · Check the mesh, then plane stress versus plane strain  *(lecture §A2, §B2)*
Quarter plate $L=1$, hole radius $R=0.1$, tension $T=1$ along $x$, $E=1$, $\nu=0.3$, P2 on curved triangles.
The first line compares the mesh area with the exact area — a cheap test of the Gmsh geometry.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from hole import *

msh, ft = plate_mesh(h_hole=0.005)
say(f"area {area(msh):.10f}   exact 1 - pi R^2/4 = {1 - np.pi * 0.01 / 4:.10f}   cells {msh.topology.index_map(2).size_global}")
for plane in ("stress", "strain"):
    _, i = solve_plate(msh, ft, isotropic(plane=plane))
    say(f"plane {plane:6s}: Kt = {i['Kt']:.4f}   max|u| = {i['umax']:.4f}   (reason {i['reason']})")

> **✏️ Your turn (5 min).** The two $K_t$ agree to three digits, the displacements differ by 10 %. Why? Which one is "right" for a thin plate?

> *your answer*

## 3 · Kirsch: the hole geometry and the mesh size  *(lecture §A4, §B3)*
Same plate; element size on the hole from $R/2.5$ to $R/40$; straight-sided triangles (`order=1`) vs curved triangles (`order=2`).

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from hole import *

iso = isotropic(); rows = []
for order in (1, 2):
    for hh in (0.04, 0.02, 0.01, 0.005, 0.0025):
        msh, ft = plate_mesh(h_hole=hh, order=order)
        _, i = solve_plate(msh, ft, iso)
        rows.append([order, hh, i["dofs"], i["Kt"]])
        say(f"order {order}  h_hole {hh:.4f}  DOFs {i['dofs']:5d}  Kt {i['Kt']:.4f}")
if comm.rank == 0:
    json.dump(rows, open("/tmp/fnx/w02_kirsch.json", "w"))

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
R = np.array(json.load(open("/tmp/fnx/w02_kirsch.json")))
fig, ax = plt.subplots(figsize=(6, 4))
for order, mk in ((1, "o-"), (2, "s-")):
    m = R[:, 0] == order
    ax.semilogx(R[m, 1], R[m, 3], mk, lw=2, label=f"geometry order {order}")
ax.axhline(3, color="k", ls="--", lw=1, label="Kirsch (infinite plate)")
ax.set_xticks([0.04, 0.02, 0.01, 0.005, 0.0025], ["R/2.5", "R/5", "R/10", "R/20", "R/40"]); ax.minorticks_off(); ax.invert_xaxis()
ax.set(xlabel="element size on the hole", ylabel="Kt"); ax.legend(); ax.grid(alpha=0.3); plt.show()

> **✏️ Your turn (5 min).** (i) Why do straight-sided triangles over-estimate $K_t$ on coarse meshes? (ii) Both curves level off near 3.085, not 3. Name
> the reason and say which computation would confirm it.

> *your answer*

## 4 · Orthotropic plate: Lekhnitskii  *(lecture §A3–A4, §B4)*
$E_1=200\,000$, $E_2=20\,000$, $\nu_{12}=0.3$, $G_{12}=10\,000$ (a fibre composite, MPa); load along $x$. $\theta$ is the angle of the stiff axis 1 from $x$.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from hole import *

E1, E2, nu12, G12 = 200e3, 20e3, 0.3, 10e3
msh, ft = plate_mesh(h_hole=0.005)
for theta, ref in ((0, lekhnitskii(E1, E2, nu12, G12)), (90, lekhnitskii(E2, E1, nu12 * E2 / E1, G12))):
    _, i = solve_plate(msh, ft, orthotropic(E1, E2, nu12, G12, np.radians(theta)))
    say(f"theta = {theta:2d} deg:  Kt FEM {i['Kt']:.4f}   Lekhnitskii (infinite plate) {ref:.4f}")

In [ ]:
%%fenicsx -np 2
# ✏️ Your turn (10 min): a code test. With E1 = E2 = 1, nu12 = 0.3 and G12 = E/(2(1+nu)), orthotropic(...) must give the same Kt as isotropic(),
# for any theta. Check theta = 0 and theta = 30 degrees on the mesh above.
# TODO

> **✏️ Your turn (5 min).** What does this test catch, and what can it **not** catch? Why is the stiff-axis value further from Lekhnitskii than the isotropic
> value was from Kirsch?

> *your answer*

## 5 · Axisymmetry: the thick cylinder  *(lecture §A5–A6, §C1–C3)*
$r_i=1$, $r_o=2$, height 1, internal pressure $p=1$, $u_z=0$ on $z=0$, top free (open ends). Mesh $n\times n/2$.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from axi import *

rows = []
for deg in (1, 2):
    for n in (4, 8, 16, 32):
        uh, i = solve_cylinder(n, deg)
        rows.append([deg, n, i["err"]])
        say(f"P{deg} n={n:2d}  DOFs {i['dofs']:5d}  rel. L2_r error of u_r {i['err']:.3e}  mean u_r(r_i) {i['ur_in']:.6f}")
say("Lame u_r(r_i):", {e: round(lame(1.0, 1.0, 2.0, 1.0, 0.0, e)[3], 6) for e in ("open", "closed", "plane_strain")})
if comm.rank == 0:
    json.dump(rows, open("/tmp/fnx/w02_lame.json", "w"))

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
R = np.array(json.load(open("/tmp/fnx/w02_lame.json")))
fig, ax = plt.subplots(figsize=(6, 4))
for deg, mk in ((1, "o-"), (2, "s-")):
    m = R[:, 0] == deg; h = 1 / R[m, 1]
    ax.loglog(h, R[m, 2], mk, lw=2, label=f"P{deg}, last rate {np.log2(R[m, 2][-2] / R[m, 2][-1]):.2f}")
ax.set_xticks(1 / np.array([4, 8, 16, 32]), ["1/4", "1/8", "1/16", "1/32"]); ax.minorticks_off()
ax.set(xlabel="h", ylabel="relative error of u_r"); ax.legend(); ax.grid(alpha=0.3); plt.show()

In [ ]:
%%fenicsx -np 2
# ✏️ Your turn (10 min): make the cylinder plane strain (u_z = 0 also on the top, ends="plane_strain" in solve_cylinder) and print the mean
# u_r(r_i) of P2 on n = 16 together with the three Lame values. Then compute the error against the OPEN-end formula on n = 8, 16, 32.
# TODO

> **✏️ Your turn (5 min).** What does the error against the wrong formula do under refinement? Why is the open-end Lamé solution an **exact** 3D solution
> of the problem in the first cell, although nothing in it is "plane stress"?

> *your answer*

## 6 · LLM exercise: a pressurised sphere in $(r,z)$  *(lecture Part D)*
Ask your LLM for a module `sphere.py` that solves a thick **spherical** shell (inner radius $a=1$, outer $b=2$, internal pressure $p$) as an
axisymmetric problem on the quarter annulus in the $(r,z)$ half-plane (Gmsh), and for a test cell. Paste the API sheet (§16) and state DOLFINx 0.11.

```text
Write a DOLFINx 0.11 module (an %%writefile /tmp/fnx/sphere.py cell) for a thick spherical shell a = 1, b = 2 under internal pressure p,
solved as an axisymmetric problem in the (r, z) half-plane: the quarter annulus a <= sqrt(r^2 + z^2) <= b, r >= 0, z >= 0, built with Gmsh on
rank 0, curved P2 triangles. Symmetry: u_r = 0 on the axis r = 0, u_z = 0 on z = 0. Functions: make_mesh(h), solve(h, p) -> (uh, info) with the
mean radial displacement on the inner surface and the converged reason. Then a %%fenicsx -np 2 test cell that compares with the Lame solution
of the sphere, u(R) = p a^3 / (E (b^3 - a^3)) * ((1 - 2 nu) R + (1 + nu) b^3 / (2 R^2)), on three meshes and prints the convergence rate.
Use the attached API sheet.
```

```python
# paste here
```

**Check before you trust it**

- [ ] DOLFINx 0.11 API: `dolfinx.io.gmsh.model_to_mesh` returning `MeshData`, `petsc_options_prefix` given
- [ ] The mesh area equals $\pi(b^2-a^2)/4$ (Gmsh boolean operations can leave extra surfaces)
- [ ] The radial displacement in the test is the **spherical** one, $\mathbf u\cdot\mathbf x/|\mathbf x|$, not $u_r$ of the cylinder coordinates
- [ ] The error falls at the rate of the element; a plateau means the wrong reference or the wrong model
- [ ] `-np 1` and `-np 2` give the same numbers